# P3 — Baseline model: logistic regression vs a bedside score

**Question:** can a model built from admission data identify children who will die — and does it beat a simple score a nurse could add up at the bedside?

**Why start with logistic regression?**
- It's the workhorse of clinical prediction (PIM, PRISM, most early-warning scores are logistic models).
- Coefficients → **odds ratios** that clinicians already understand.
- It sets the **baseline**: any fancy model (P5) must beat it to justify its complexity.

**Plan**
1. Choose predictors *clinically*, respecting EPV
2. Build a `Pipeline` (impute → scale → model), so preprocessing is learned **inside** each training fold
3. Build a bedside danger-sign score (no fitting)
4. Compare both with **repeated cross-validation**: AUROC, AUPRC, and the challenge metric (TPR at FPR ≤ 20%)
5. Read the odds ratios
6. The "kitchen sink": what happens when we throw in all 136 columns?

⚠️ Synthetic data — methods transfer, exact numbers may not.

## 1. Load the P2 feature table
**What to look for:** 2,686 children, 119 deaths (4.4%).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve, precision_recall_curve
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = pd.read_csv("p2_features.csv")
y = X.pop("inhospital_mortality").to_numpy()
print(X.shape, "| deaths:", y.sum(), f"({y.mean():.1%})")

## 2. Choose predictors — clinically, not by p-value
With **119 deaths** and EPV ≥ 10, we can afford about **11 parameters**. We pick features with a clear clinical rationale (sepsis severity, nutrition, key comorbidity):

| Domain | Predictor |
|---|---|
| Neurological | `bcs_total` (Blantyre 0–5) |
| Nutrition | `waz` (weight-for-age z) |
| Perfusion | `lactate`, `lactate_measured`, `caprefill` (≥ 3 s) |
| Respiratory | `spo2_min`, `rr_ratio` (RR ÷ IMCI cut-off), `respdistress` |
| Comorbidity / cause | `hiv_pos`, `malaria_pos` |
| General | `not_feeding` (not feeding at all) |

⚠️ **Avoid:** picking the predictors with the lowest p-values in the whole dataset, and then testing on the same data. That's selection leakage, and it inflates performance.

We keep missing values as NaN for the binary signs; the pipeline will impute them.

In [ ]:
yes = lambda col, val: (X[col] == val).astype(float).where(X[col].notna())
X["caprefill"] = yes("caprefill_adm", "Yes")
X["respdistress"] = yes("respdistress_adm", "Yes")
X["hiv_pos"] = yes("hivstatus_adm", "HIV positive")
X["malaria_pos"] = yes("malariastatuspos_adm", "Yes")
X["not_feeding"] = (X["feedingstatus_adm"] == "Not feeding at all").astype(float)

NUM = ["bcs_total", "waz", "lactate_mmolpl_adm", "spo2_min", "rr_ratio"]
BIN = ["lactate_measured", "caprefill", "respdistress", "hiv_pos", "malaria_pos", "not_feeding"]
PRED = NUM + BIN
print(f"{len(PRED)} predictors  ->  EPV = {y.sum()} / {len(PRED)} = {y.sum() / len(PRED):.1f}")
X[PRED].describe().T.round(2)

## 3. The Pipeline — why preprocessing lives *inside* the model
A `Pipeline` chains the steps so that `fit()` learns **everything** from the training data only:

```
numeric → median imputation → standardise (mean 0, SD 1) ┐
binary  → most-frequent imputation ─────────────────────┴→ logistic regression
```

If you compute the median or the SD on the **whole** dataset first, the test children have already influenced training. That's a quiet form of leakage. With a Pipeline, cross-validation re-learns the medians and SDs inside every fold automatically.

`C=1e6` means essentially **no penalty** (classic logistic regression). We'll change that in §7.

In [ ]:
prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUM),
    ("bin", SimpleImputer(strategy="most_frequent"), BIN)])

model = Pipeline([("prep", prep), ("lr", LogisticRegression(C=1e6, max_iter=5000))])
model

## 4. The bedside score — the comparator
A model is only useful if it beats what clinicians can already do. We build a **danger-sign count**: 1 point each for

- BCS ≤ 2 (coma)
- severe acute malnutrition
- SpO₂ < 90%
- capillary refill ≥ 3 s
- severe respiratory distress
- not feeding at all

Score range 0–6. **No fitting, no data needed** — so it can't overfit. (This mirrors WHO ETAT emergency signs.)

**What to look for:** a clear gradient — about 2% mortality with 0 signs, 12% with 2, and 23% with 4. The groups with 5–6 points are tiny (5 children), so ignore their percentages.

In [ ]:
bedside = (X["bcs_coma"] + X["sam"] + X["hypoxaemia"] + X["caprefill"].fillna(0)
           + X["respdistress"].fillna(0) + X["not_feeding"])

pd.DataFrame({"children": bedside.value_counts().sort_index(),
              "deaths": pd.Series(y).groupby(bedside).sum(),
              "mortality_%": (pd.Series(y).groupby(bedside).mean() * 100).round(1)})

## 5. Repeated cross-validation: model vs bedside score
**Why cross-validation?** With only 119 deaths, a single 80/20 split puts about 24 deaths in the test set, and one lucky or unlucky split swings AUROC by ±0.05. **5-fold × 10 repeats** = 50 test sets, so we can report a mean *and* its spread.

**Three metrics:**
| Metric | Question it answers | "Useless" value |
|---|---|---|
| **AUROC** | Does a random death get a higher score than a random survivor? | 0.5 |
| **AUPRC** | Among the children we flag, how many actually die (averaged over thresholds)? More honest with rare outcomes | prevalence = 0.044 |
| **TPR @ FPR ≤ 20%** | The challenge metric: if we accept false alarms in 20% of survivors, what fraction of deaths do we catch? | 0.20 |

**What to look for:**
- Logistic ≈ 0.75 AUROC vs bedside ≈ 0.72: the model is better, but not by much. **The overlap in SD matters.**
- AUPRC ≈ 0.22 vs 0.13: that's 5× better than chance, but it also means most flagged children survive (P4).
- The model catches about 60% of deaths at the 20% false-alarm budget. The score catches about 52%.

*(This takes about 20 seconds.)*

In [ ]:
def tpr_at_fpr(y_true, score, max_fpr=0.20):
    fpr, tpr, _ = roc_curve(y_true, score)
    return tpr[fpr <= max_fpr].max()

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
rows = []
for tr, te in cv.split(X, y):
    p = model.fit(X.iloc[tr][PRED], y[tr]).predict_proba(X.iloc[te][PRED])[:, 1]
    for name, s in [("logistic (11)", p), ("bedside score", bedside.iloc[te].to_numpy())]:
        rows.append([name, roc_auc_score(y[te], s), average_precision_score(y[te], s), tpr_at_fpr(y[te], s)])

res = pd.DataFrame(rows, columns=["model", "AUROC", "AUPRC", "TPR@FPR<=20%"])
res.groupby("model").agg(["mean", "std"]).round(3)

## 6. Seeing it: ROC and precision–recall curves
We use **out-of-fold** predictions: every child is scored by a model that never saw them (`cross_val_predict`).

**What to look for:**
- **ROC (left):** both curves bend above the diagonal. The dashed line marks the FPR = 20% budget. The bedside score is a "staircase" because it only has 7 possible values.
- **PR (right):** precision starts high for the handful of highest-risk children, then drops fast towards the 4.4% baseline. This is the reality of rare outcomes: **to catch most deaths, you'll alarm on many survivors.**

In [ ]:
oof = cross_val_predict(model, X[PRED], y, cv=StratifiedKFold(5, shuffle=True, random_state=1),
                        method="predict_proba")[:, 1]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
for name, s in [("logistic (11)", oof), ("bedside score", bedside.to_numpy())]:
    fpr, tpr, _ = roc_curve(y, s)
    ax[0].plot(fpr, tpr, label=f"{name}  AUROC {roc_auc_score(y, s):.2f}")
    prec, rec, _ = precision_recall_curve(y, s)
    ax[1].plot(rec, prec, label=f"{name}  AUPRC {average_precision_score(y, s):.2f}")
ax[0].plot([0, 1], [0, 1], c="grey", lw=.8); ax[0].axvline(.2, ls="--", c="grey")
ax[0].set(xlabel="False-positive rate (1 - specificity)", ylabel="True-positive rate (sensitivity)", title="ROC")
ax[1].axhline(y.mean(), ls="--", c="grey", label=f"chance = {y.mean():.3f}")
ax[1].set(xlabel="Recall (sensitivity)", ylabel="Precision (PPV)", title="Precision–recall")
for a in ax: a.legend()
plt.tight_layout(); plt.show()

## 7. Odds ratios — what the model learned
We refit on **all** the data (this is the model you would publish) and exponentiate the coefficients.
- **Numeric predictors:** OR per **1 SD** increase (because we standardised).
- **Binary predictors:** OR for yes vs no.

**What to look for:**
- **Blantyre score OR ≈ 0.56 per SD**: each step towards full consciousness roughly halves the odds. It's the strongest predictor.
- **HIV+ ≈ 2.2, severe respiratory distress ≈ 2.0, lower WAZ** (OR 0.70 per SD higher) → all clinically coherent.
- **Malaria+ ≈ 0.65**: protective *association*, as in P1 (a treatable cause). It is not causal.
- Lactate, SpO₂ and RR have ORs **≈ 1** once the other predictors are in the model. Their information overlaps with BCS and respiratory distress. (SpO₂ OR 1.07 in the "wrong" direction is noise; P5 will add confidence intervals.)
- **Apparent (training) AUROC 0.78 > CV AUROC 0.75**: that's the optimism gap. Always report the cross-validated number.

In [ ]:
model.fit(X[PRED], y)
odds = np.exp(pd.Series(model["lr"].coef_[0], index=PRED)).sort_values()

ax = odds.plot.barh(figsize=(6, 4.5)); ax.axvline(1, c="k", lw=.8); ax.set_xscale("log")
ax.set_xlabel("Odds ratio (log scale)  —  <1 protective, >1 harmful"); plt.tight_layout(); plt.show()
print(odds.round(2).to_string())
print("\nApparent (training) AUROC:", round(roc_auc_score(y, model.predict_proba(X[PRED])[:, 1]), 3))

## 8. The "kitchen sink": every column at once
What if we skip the clinical thinking and feed in **all** columns (one-hot encoded)? That gives about **336 parameters for 119 deaths → EPV 0.35**.

Two versions:
- **No penalty** (C = 10⁶): classic logistic regression.
- **L2 penalty** (C = 0.01): *ridge* regression. It shrinks every coefficient towards 0, so the model can't lean hard on any single noisy column.

**What to look for:**
- No penalty: train AUROC ≈ 0.96, CV ≈ 0.66. That's **memorisation**: the model has learned the training children, not the disease.
- L2: train ≈ 0.87, **CV ≈ 0.78**, which is *better* than our hand-picked 11 (0.75)! Penalisation lets many weak signals add up without overfitting.

*(Takes about 30 seconds.)*

In [ ]:
allnum = X.select_dtypes("number").columns.tolist()
allcat = X.select_dtypes(exclude="number").columns.tolist()
sink_prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), allnum),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=10))]), allcat)])
n_param = sink_prep.fit_transform(X).shape[1]
print(f"{n_param} parameters, EPV = {y.sum() / n_param:.2f}\n")

for name, C in [("no penalty", 1e6), ("L2, C=0.01", 0.01)]:
    sink = Pipeline([("prep", sink_prep), ("lr", LogisticRegression(C=C, max_iter=5000))])
    tr_auc, cv_auc = [], []
    for tr, te in RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=0).split(X, y):
        sink.fit(X.iloc[tr], y[tr])
        tr_auc.append(roc_auc_score(y[tr], sink.predict_proba(X.iloc[tr])[:, 1]))
        cv_auc.append(roc_auc_score(y[te], sink.predict_proba(X.iloc[te])[:, 1]))
    print(f"{name:12s}  train AUROC {np.mean(tr_auc):.3f}  |  CV AUROC {np.mean(cv_auc):.3f}")

### 8b. So should we use the kitchen sink? Look at *what* it leans on
Higher AUROC is not the whole story. Look at the biggest coefficients.

**What to look for:** alongside sensible features (BCS, MUAC, respiratory distress, HIV), the model uses
- `spo2onoxy` (was SpO₂ measured **on oxygen**?): a **treatment** marker, which means clinicians were already worried.
- `bcgscar` (BCG vaccine scar): a proxy for **access to care** and socioeconomic status.
- `sqi2_perc_oxi` (oximeter **signal quality**): a property of the *recording*, not the child.

These predict well **here**, but they may not transport to a hospital with different oxygen policies, vaccination coverage or devices. A 136-item model is also impractical to fill in at triage.

**The trade-off:** accuracy vs parsimony, transportability, and usability. There's no free lunch. We revisit this in P5 (trees, SHAP) and P6 (transportability).

In [ ]:
sink = Pipeline([("prep", sink_prep), ("lr", LogisticRegression(C=0.01, max_iter=5000))]).fit(X, y)
co = pd.Series(sink["lr"].coef_[0], index=sink["prep"].get_feature_names_out())
co[co.abs().sort_values(ascending=False).index[:15]].round(2)

## Summary
| Model | Parameters | CV AUROC | Comment |
|---|---|---|---|
| Bedside danger-sign count | 0 (no fitting) | ~0.72 | Transparent, zero cost, can't overfit |
| Logistic, 11 clinical predictors | 11 | ~0.75 | Interpretable ORs; modest gain |
| Kitchen sink, no penalty | ~336 | ~0.66 | Overfits (train 0.96) |
| Kitchen sink, L2 | ~336 | ~0.78 | Best number, but leans on proxies |

**Key messages**
1. Always compare against a **simple clinical baseline**.
2. Preprocessing goes **inside** the Pipeline, so it's refit in each fold.
3. Report **cross-validated** performance with its **spread**; apparent performance is optimistic.
4. With rare outcomes, look at **AUPRC** and the operating point, not just AUROC.
5. Regularisation beats EPV limits, but check *what* the model uses.

We have **not** yet asked whether the predicted probabilities are *correct* (calibration), or chosen a threshold. That's P4.

## ✍️ Try it yourself
1. **Log-lactate:** lactate is right-skewed. Replace it with `np.log(X["lactate_mmolpl_adm"])` in `NUM`. Does the OR or the AUROC change?
2. **Drop the weak ones:** remove `spo2_min`, `rr_ratio` and `lactate_mmolpl_adm` (ORs ≈ 1). Does CV AUROC drop? (Fewer predictors = simpler bedside use.)
3. **Tune the penalty:** in §8 try `C = 0.003, 0.03, 0.1`. Which is best? (Hint: we saw 0.785 / 0.773 / 0.757.) Choosing C *using the test folds* is itself leakage. The proper way is nested CV or `LogisticRegressionCV` inside the Pipeline.
4. **Clinician's check:** from the §4 table, what sensitivity would "flag if bedside score ≥ 2" give? Compute it: `((bedside >= 2) & (y == 1)).sum() / y.sum()`.

**Next — P4:** class imbalance, calibration (are the risks right?), choosing a threshold at FPR ≤ 20%, and decision curves.